# Predicción de aristas — ¿ayuda a decir a quién favorece una cláusula y qué exposición genera?

La propuesta del experto: con *link prediction* sobre el grafo, cada arista que se predice ya
viene con su significado, porque los tipos de relación están definidos de antemano. Sugirió
modelos de embeddings de grafo del tipo TransE, y TransH o TransR para las relaciones de
muchos a uno.

Este cuaderno lo pone a prueba sobre los tres grafos fiables —SteelVault para desarrollar,
Ediets y Bellicum para comprobar— **solo con los tipos de arista que ya existen**, y siempre
contra líneas base baratas. Al final enseña cómo pasar de las aristas, observadas o predichas,
a «esta cláusula favorece a X y genera esta exposición» sin pesos inventados.

**Qué no hace.** No toca el grafo (`infra/json/kg/`) ni el pipeline de extracción. Lo que
predice se guarda aparte, en `infra/json/link_prediction/`. No llama a ningún modelo de pago:
el coste de usar uno está estimado en la sección 6 y espera aprobación.

**Para ejecutarlo** hace falta el grupo de dependencias `link-prediction` del servidor
(PyKEEN, PyTorch, scikit-learn y sentence-transformers):

```
cd server && uv sync --group link-prediction
```

Los experimentos largos se guardan en `infra/json/link_prediction/results.json` con la huella
de los grafos, de la configuración y del código que los produjo; si algo cambia, se repiten.
`RECOMPUTE = True` los repite siempre. Todo corre en local: con GPU tarda unas tres horas la
primera vez —casi todo es el ajuste de los modelos y Bellicum—; luego, un par de minutos.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import torch
from IPython.display import Markdown, display

import notebooks.link_prediction as lp

RECOMPUTE = False
torch.set_num_threads(8)
contracts = {name: lp.load_contract(name) for name in lp.CONTRACTS}
kg_prints = {name: lp.fingerprint(lp.KG_DIR / f"{c.stem}.json") for name, c in contracts.items()}
print("dispositivo para los modelos de grafo:", lp.DEVICE, "| huellas de los grafos:", kg_prints)

dispositivo para los modelos de grafo: cuda | huellas de los grafos: {'SteelVault': '117bf65f5716', 'Ediets': '08147d413a8d', 'Bellicum': 'd6dc9f75a905'}


## 1. Las tres tareas, y qué aporta cada una a la pregunta

| Tarea | Aristas | Qué se oculta y qué se predice | Qué aporta a la pregunta |
|---|---|---|---|
| **T1** atribución de partes | `assigns_obligation_to`, `grants_right_to` | se oculta la parte de un enunciado y se adivina entre las del contrato | la **dirección** de todo: quién carga y quién se beneficia. Sin ella no hay reparto ni exposición. La extracción ya la da; T1 serviría de **segunda opinión** para auditarla |
| **T2** cadenas de exposición | `depends_on` de la consecuencia de un incumplimiento al deber incumplido | dada la consecuencia (lo que abre la condición), se adivina qué deber se incumplió | la **exposición al riesgo** como camino: deber de A → incumplimiento → derecho o cobro de B → cuantía o plazo. Hoy el grafo tiene la consecuencia y el texto del disparador, pero no el deber al que apunta |
| **T3** relaciones entre cláusulas | `depends_on`, `modifies`, `references` | se oculta una relación y se adivina la cláusula de destino | qué cláusula **recorta o condiciona** a otra: una exclusión que neutraliza una garantía cambia a quién favorece la garantía. También da al PageRank las aristas informativas que le faltan |

T2 es la única que añade algo que el grafo no tiene: T1 y T3 completan o revisan lo que la
extracción ya produce.

## 2. Lo que los grafos permiten, y lo que no

Antes de elegir modelo hay que mirar qué forma tienen los datos.

In [2]:
rows = []
for name, c in contracts.items():
    party = lp.party_queries(c)
    informative = lp.clause_queries(c)
    st_to_st = sum(1 for h, r, t in c.triples if h in c.statements and t in c.statements)
    same = sum(1 for s in c.statements.values() if s.get("burdenPartyId") and s.get("burdenPartyId") == s.get("benefitPartyId"))
    rows.append({
        "contrato": name, "partes": len(c.parties), "cláusulas": len(c.clauses), "enunciados": len(c.statements),
        "aristas": len(c.triples), "aristas de parte": len(party),
        "de ellas, gemelos bilaterales": sum(1 for q in party if q[0] in c.twin),
        "depends_on": sum(1 for q in informative if q[1] == "depends_on"),
        "modifies": sum(1 for q in informative if q[1] == "modifies"),
        "references": sum(1 for q in informative if q[1] == "references"),
        "enunciado→enunciado": st_to_st, "obligado = beneficiario": same,
    })
lp.table(rows)

| contrato | partes | cláusulas | enunciados | aristas | aristas de parte | de ellas, gemelos bilaterales | depends_on | modifies | references | enunciado→enunciado | obligado = beneficiario |
|---|---|---|---|---|---|---|---|---|---|---|---|
| SteelVault | 2 | 26 | 90 | 275 | 83 | 36 | 0 | 6 | 1 | 0 | 0 |
| Ediets | 2 | 68 | 300 | 1202 | 298 | 146 | 13 | 43 | 20 | 0 | 0 |
| Bellicum | 2 | 120 | 631 | 2867 | 610 | 240 | 52 | 101 | 60 | 0 | 0 |

Cuatro hechos deciden el método:

- **Dos partes por contrato.** Atribuir una parte es elegir entre dos. En esa situación el MRR
  es `(1 + acierto) / 2` y Hits@2 vale siempre 100%: no dicen nada que no diga el acierto. En
  T1 se informa el acierto. Y como ningún enunciado tiene a la misma parte de obligada y de
  beneficiaria, conocer una da la otra: `benefitPartyId` no es una tarea aparte.
- **Casi la mitad de las aristas de parte son gemelos bilaterales**: el mismo texto, en la
  misma cláusula, emitido una vez por parte. Nada distingue a un gemelo del otro, ni el texto
  ni el grafo, y su parte la fija la extracción por construcción. Por eso **no se ponen a
  prueba**: se quedan en el grafo como contexto conocido y T1 se mide sobre los enunciados
  unilaterales.
- **Ninguna arista une dos enunciados** (columna enunciado→enunciado). Para T2 no hay ni un
  ejemplo del que aprender: hace falta anotarlo a mano (sección 6).
- **SteelVault tiene 7 aristas informativas.** T3 se mide sobre Ediets y Bellicum, con una
  salvedad: allí las aristas las produjo la extracción y nadie las ha revisado, así que se
  mide cuánto se reproduce lo extraído, no cuánto se acierta.

En SteelVault, en cambio, las partes sí están verificadas: el grafo acierta la parte del 100%
de los enunciados de la referencia anotada a mano
([`evaluate_kg.ipynb`](evaluate_kg.ipynb)), así que sus aristas de parte son la verdad contra
la que medir T1.

## 3. Los modelos

| Familia | Qué supone | ¿Encaja aquí? | ¿Se mide? |
|---|---|---|---|
| **TransE** | una relación es una traslación: origen + relación ≈ destino | con decenas de enunciados apuntando a la misma parte, empuja a todos hacia el mismo punto: no puede separarlos por nada más | sí |
| **TransH, TransR** | proyectan las entidades a un plano o espacio propio de cada relación antes de trasladar | nacieron para el muchos-a-uno: el enunciado puede ser distinto en general y coincidir con su parte solo en la proyección | sí |
| **RotatE, DistMult, ComplEx** | rotación en complejos; producto trilineal (DistMult simétrico, ComplEx no) | modelos de referencia; DistMult no distingue origen de destino, pero aquí el tipo ya lo fija | sí |
| **R-GCN, CompGCN** | redes que pasan mensajes por el grafo antes de puntuar | el vector de un enunciado se construye con sus vecinos (cláusula, términos, condiciones) | sí |
| **StarE** | hechos con calificadores: (enunciado, carga a, parte) con {condición, plazo, valor} | el grafo ya está *reificado* —el enunciado es un nodo y la condición y el valor cuelgan de él— así que lo que StarE añade lo ven ya R-GCN y CompGCN. Encaja mejor en el trabajo futuro, entre contratos, con roles abstractos | no |
| **GraIL** | inductivo: puntúa el subgrafo que rodea al par origen-destino | el destino de T1 es una parte unida a casi todo, y su subgrafo es casi el grafo entero | no |
| **NBFNet** | inductivo: suma sobre todos los caminos entre origen y destino (Bellman-Ford generalizado) y explica con caminos | no guarda un vector por entidad, así que lo aprendido vale para cualquier nodo | sí, implementación mínima en `link_prediction.py` |
| **ULTRA** | NBFNet preentrenado en muchos grafos, sin entrenar en el nuevo | promete justo lo que aquí falta, generalizar entre contratos; necesita sus pesos y su código (PyTorch Geometric y un núcleo compilado) | no: misma familia que NBFNet; si NBFNet entrenado en el propio grafo no supera a las reglas, no hay motivo para esperarlo de ULTRA sin entrenar. **Es una inferencia, no una medida** |

Todos los de PyKEEN son **transductivos**: aprenden un vector por entidad, y el «party-2» de
un contrato no es el de otro. Se entrenan y se prueban dentro de cada contrato.

### Configuración

Cada modelo se ajusta en SteelVault con una rejilla pequeña —dos formas de entrenar, tasa de
aprendizaje, épocas, dimensión— y se elige por cómo predice un 10% de aristas de cualquier tipo
que no ve. Así la configuración no mira las tareas.

In [3]:
SV = contracts["SteelVault"]
tuning = lp.cached(
    "ajuste:SteelVault",
    lambda: lp.tune_kge(SV),
    inputs={"kg": kg_prints["SteelVault"], "grid": lp.TUNING_GRID, "seeds": [0, 1]},
    depends=[lp.tune_kge, lp.kge_validation_mrr, lp.train_kge, lp.entity_ids, lp.load_contract],
    recompute=RECOMPUTE,
)
KGE = {m: {k: v for k, v in cfg.items() if k != "MRR validación"} for m, cfg in tuning["best"].items()}
lp.table([{"modelo": m, **cfg} for m, cfg in tuning["best"].items()])

| modelo | loss | lr | epochs | embedding_dim | MRR validación |
|---|---|---|---|---|---|
| TransE | 1vsall | 0.030 | 100 | 128 | 0.465 |
| TransH | margin | 0.100 | 300 | 32 | 0.228 |
| TransR | margin | 0.030 | 100 | 32 | 0.115 |
| RotatE | 1vsall | 0.010 | 300 | 128 | 0.454 |
| DistMult | 1vsall | 0.100 | 300 | 128 | 0.403 |
| ComplEx | margin | 0.100 | 100 | 128 | 0.175 |
| RGCN | 1vsall | 0.030 | 300 | 128 | 0.446 |
| CompGCN | 1vsall | 0.100 | 300 | 32 | 0.424 |

Con lo que el grafo enseña de sí mismo, TransH, TransR y ComplEx quedan muy por debajo de TransE.
Para descartar un fallo del bucle de entrenamiento, el mismo bucle sobre *Nations*, el grafo de
prueba pequeño que trae PyKEEN (14 países, 55 relaciones, sin descargas):

In [4]:
import collections

from pykeen.datasets import Nations

nations = Nations()
train_n = [tuple(x) for x in nations.training.triples]
test_n = [tuple(x) for x in nations.testing.triples]
e2i_n, r2i_n = nations.training.entity_to_id, nations.training.relation_to_id
known_n = collections.defaultdict(set)
for h, r, t in train_n + test_n + [tuple(x) for x in nations.validation.triples]:
    known_n[(h, r)].add(e2i_n[t])
rows = []
for m in ("TransE", "TransH", "TransR", "ComplEx"):
    model = lp.train_kge(m, train_n, e2i_n, r2i_n, seed=0, epochs=200, lr=0.01, embedding_dim=64)
    scores = lp.kge_tail_scores(model, [(h, r) for h, r, _ in test_n], e2i_n, r2i_n)
    rr = []
    for row, (h, r, t) in zip(scores, test_n):
        row, target = row.copy(), e2i_n[t]
        row[[i for i in known_n[(h, r)] if i != target]] = -np.inf
        rr.append(1 / (1 + np.sum(row > row[target])))
    rows.append({"modelo": m, "MRR en Nations": float(np.mean(rr))})
lp.table(rows)

| modelo | MRR en Nations |
|---|---|
| TransE | 0.451 |
| TransH | 0.459 |
| TransR | 0.386 |
| ComplEx | 0.536 |

En *Nations* los cuatro aprenden: no es el bucle. Es el grafo del contrato: casi todos sus nodos
tienen una, dos o tres aristas, y una proyección o un espacio complejo por relación tiene más
parámetros que datos para ajustarlos.

## 4. T1 — atribución de partes

**Protocolo.** Validación cruzada en cinco pliegues sobre las aristas de parte de los
enunciados unilaterales. En cada pliegue se ocultan sus aristas y se entrena con el resto del
grafo; los gemelos siguen visibles. En SteelVault se repite con tres repartos distintos de los
pliegues; la tabla da la media, la peor y la mejor repetición, y el intervalo de Wilson al 95%
sobre todas las predicciones. En Ediets y Bellicum, una repetición.

**Líneas base**, de más tonta a menos:

- *mayoría*: la parte que más aparece en ese tipo de arista;
- *cláusula*: la mayoría de su propia cláusula (o de la madre, si la suya no tiene ninguna);
- *sujeto*: la parte que el texto nombra primero —en «Equidata reserves the right…» el titular
  abre la frase—, y si no nombra a ninguna, la de *cláusula*;
- *texto*: una regresión logística y el vecino más cercano sobre el embedding MiniLM del
  enunciado, entrenados con las partes conocidas del mismo contrato;
- *gemelo→…*: si el gemelo es conocido, la otra parte. Con los unilaterales no cambia nada:
  está para la comprobación de más abajo.

La extracción misma es la línea base con LLM: ya acierta el 100% en SteelVault, así que no
hay nada que pagar para medirla.

In [5]:
t1 = {}
for name, repeats in (("SteelVault", 3), ("Ediets", 1), ("Bellicum", 1)):
    c = contracts[name]
    t1[name] = lp.cached(
        f"T1:{name}",
        lambda c=c, repeats=repeats: lp.run_t1(c, KGE, repeats=repeats, k=5, nbfnet_epochs=10),
        inputs={"kg": kg_prints[name], "kge": KGE, "repeats": repeats, "k": 5, "nbfnet_epochs": 10},
        depends=lp.T1_CODE,
        recompute=RECOMPUTE,
    )
for name in t1:
    display(Markdown(f"**{name}** — {len(t1[name]['queries'])} enunciados unilaterales"))
    display(lp.table(lp.t1_summary(t1[name]), ["método", "acierto", "peor", "mejor", "IC95 bajo", "IC95 alto"]))

**SteelVault** — 47 enunciados unilaterales

| método | acierto | peor | mejor | IC95 bajo | IC95 alto |
|---|---|---|---|---|---|
| sujeto | 0.865 | 0.851 | 0.872 | 0.799 | 0.912 |
| gemelo→sujeto→cláusula | 0.865 | 0.851 | 0.872 | 0.799 | 0.912 |
| cláusula | 0.858 | 0.830 | 0.872 | 0.791 | 0.906 |
| gemelo→cláusula | 0.858 | 0.830 | 0.872 | 0.791 | 0.906 |
| texto: regresión logística | 0.794 | 0.766 | 0.809 | 0.720 | 0.853 |
| RotatE | 0.766 | 0.745 | 0.809 | 0.690 | 0.828 |
| texto: vecino más cercano | 0.745 | 0.702 | 0.787 | 0.667 | 0.809 |
| mayoría | 0.745 | 0.745 | 0.745 | 0.667 | 0.809 |
| NBFNet | 0.745 | 0.745 | 0.745 | 0.667 | 0.809 |
| TransE | 0.660 | 0.596 | 0.766 | 0.578 | 0.733 |
| RGCN | 0.596 | 0.574 | 0.617 | 0.513 | 0.673 |
| TransR | 0.574 | 0.447 | 0.766 | 0.492 | 0.653 |
| TransH | 0.553 | 0.489 | 0.638 | 0.471 | 0.633 |
| CompGCN | 0.553 | 0.511 | 0.596 | 0.471 | 0.633 |
| DistMult | 0.525 | 0.426 | 0.574 | 0.443 | 0.605 |
| ComplEx | 0.504 | 0.447 | 0.532 | 0.422 | 0.585 |

**Ediets** — 152 enunciados unilaterales

| método | acierto | peor | mejor | IC95 bajo | IC95 alto |
|---|---|---|---|---|---|
| sujeto | 0.757 | 0.757 | 0.757 | 0.683 | 0.818 |
| gemelo→sujeto→cláusula | 0.757 | 0.757 | 0.757 | 0.683 | 0.818 |
| texto: regresión logística | 0.691 | 0.691 | 0.691 | 0.613 | 0.759 |
| cláusula | 0.612 | 0.612 | 0.612 | 0.533 | 0.686 |
| gemelo→cláusula | 0.612 | 0.612 | 0.612 | 0.533 | 0.686 |
| DistMult | 0.612 | 0.612 | 0.612 | 0.533 | 0.686 |
| RotatE | 0.599 | 0.599 | 0.599 | 0.519 | 0.673 |
| texto: vecino más cercano | 0.592 | 0.592 | 0.592 | 0.513 | 0.667 |
| TransH | 0.572 | 0.572 | 0.572 | 0.493 | 0.648 |
| TransE | 0.566 | 0.566 | 0.566 | 0.486 | 0.642 |
| mayoría | 0.553 | 0.553 | 0.553 | 0.473 | 0.629 |
| RGCN | 0.553 | 0.553 | 0.553 | 0.473 | 0.629 |
| CompGCN | 0.553 | 0.553 | 0.553 | 0.473 | 0.629 |
| TransR | 0.526 | 0.526 | 0.526 | 0.447 | 0.604 |
| NBFNet | 0.513 | 0.513 | 0.513 | 0.434 | 0.591 |
| ComplEx | 0.480 | 0.480 | 0.480 | 0.402 | 0.559 |

**Bellicum** — 370 enunciados unilaterales

| método | acierto | peor | mejor | IC95 bajo | IC95 alto |
|---|---|---|---|---|---|
| sujeto | 0.757 | 0.757 | 0.757 | 0.711 | 0.798 |
| gemelo→sujeto→cláusula | 0.757 | 0.757 | 0.757 | 0.711 | 0.798 |
| texto: regresión logística | 0.751 | 0.751 | 0.751 | 0.705 | 0.793 |
| NBFNet | 0.676 | 0.676 | 0.676 | 0.626 | 0.721 |
| texto: vecino más cercano | 0.670 | 0.670 | 0.670 | 0.621 | 0.716 |
| RotatE | 0.657 | 0.657 | 0.657 | 0.607 | 0.703 |
| cláusula | 0.651 | 0.651 | 0.651 | 0.601 | 0.698 |
| gemelo→cláusula | 0.651 | 0.651 | 0.651 | 0.601 | 0.698 |
| TransE | 0.635 | 0.635 | 0.635 | 0.585 | 0.683 |
| DistMult | 0.611 | 0.611 | 0.611 | 0.560 | 0.659 |
| ComplEx | 0.576 | 0.576 | 0.576 | 0.525 | 0.625 |
| RGCN | 0.535 | 0.535 | 0.535 | 0.484 | 0.585 |
| TransH | 0.532 | 0.532 | 0.532 | 0.482 | 0.583 |
| CompGCN | 0.519 | 0.519 | 0.519 | 0.468 | 0.569 |
| TransR | 0.514 | 0.514 | 0.514 | 0.463 | 0.564 |
| mayoría | 0.476 | 0.476 | 0.476 | 0.425 | 0.527 |

**Lectura.** En los tres contratos gana la regla del sujeto (86,5%, 75,7% y 75,7%), y en
SteelVault su peor repetición (85,1%) queda por encima de la mejor de cualquier modelo de grafo
(RotatE, 80,9%). Ningún modelo de grafo supera con claridad a la mayoría de su propia cláusula.
TransH y TransR, los pensados para el muchos-a-uno, quedan por debajo de TransE en SteelVault y
en Bellicum.

El muchos-a-uno no es el cuello de botella. Lo es que un enunciado tiene, además de su arista de
parte, una, dos o tres aristas más, y ninguna dice quién carga con él. Eso lo dice el texto, y
quien mejor lo lee es la extracción, que en SteelVault acierta todas.

### Los gemelos enseñan lo contrario

Si NBFNet aprende también de los gemelos, aprende una regla falsa: «la parte de este enunciado
es la contraria a la de su vecino idéntico». Como el grafo no tiene ninguna arista que diga que
dos enunciados son la misma disposición, no puede distinguir a un gemelo de un vecino cualquiera
de su cláusula, y aplica la regla a todos.

In [6]:
poisoned = lp.cached(
    "T1:SteelVault:NBFNet con gemelos",
    lambda: lp.run_t1(SV, {}, repeats=3, k=5, nbfnet_epochs=10, twins_as_nbfnet_queries=True),
    inputs={"kg": kg_prints["SteelVault"], "repeats": 3, "k": 5, "nbfnet_epochs": 10, "twins": True},
    depends=lp.T1_CODE,
    recompute=RECOMPUTE,
)
rows = [r for r in lp.t1_summary(t1["SteelVault"]) if r["método"] in ("NBFNet", "mayoría")]
rows += [dict(r, método="NBFNet aprendiendo también de los gemelos") for r in lp.t1_summary(poisoned) if r["método"] == "NBFNet"]
lp.table(rows, ["método", "acierto", "peor", "mejor"])

| método | acierto | peor | mejor |
|---|---|---|---|
| mayoría | 0.745 | 0.745 | 0.745 |
| NBFNet | 0.745 | 0.745 | 0.745 |
| NBFNet aprendiendo también de los gemelos | 0.681 | 0.617 | 0.723 |

### Lo que T1 no puede hacer: rellenar los huecos

Siete enunciados de SteelVault no tienen parte. La referencia anotada dice por qué: cuatro son
deberes de los árbitros, uno es el avalista personal, otro es la instrucción del bloque de
firmas y solo uno —*«all monies determined owed…»*— podría atribuirse, y la referencia lo marca
como discutible. Un predictor que elige entre dos partes asigna una siempre: en seis de los
siete casos, la respuesta correcta es «ninguna».

In [7]:
gold = json.loads((lp.GOLD_DIR / f"{SV.stem}.json").read_text())
gold_of = {v: k for k, v in lp.match_gold_statements(SV, gold).items()}
gold_by_id = {g["id"]: g for g in gold["statements"]}
nulls = [sid for sid in SV.statements if SV.party_edge(sid) is None]
known = [q for q in lp.party_queries(SV)]
fake = [(sid, "assigns_obligation_to", None) for sid in nulls]
guess = lp.t1_baselines(SV, known, fake)["sujeto"]
rows = []
for sid in nulls:
    g = gold_by_id.get(gold_of.get(sid), {})
    flag = "árbitros" if g.get("nonParty") else "avalista" if g.get("burdenAlso") else "discutible" if g.get("optional") else "sin referencia" if not g else ""
    rows.append({"enunciado": sid, "texto": lp.statement_text(SV.statements[sid])[:80], "regla del sujeto diría": SV.parties[guess[sid]]["name"] if guess[sid] else "—", "referencia": flag or "parte del contrato"})
lp.table(rows)

| enunciado | texto | regla del sujeto diría | referencia |
|---|---|---|---|
| obligation-13 | the undersigned principal, partner or owner will become responsible for any unpa | National Credit Report.com, LLC | avalista |
| obligation-19 | all monies determined owed shall be considered due and payable immediately | National Credit Report.com, LLC | discutible |
| obligation-29 | the two arbitrators will jointly select a third arbitrator(s). | National Credit Report.com, LLC | árbitros |
| obligation-30 | Not later than 5 calendar days after the arbitrator(s) have been selected, the a | National Credit Report.com, LLC | árbitros |
| obligation-31 | The hearing shall commence no later than 25 calendar days after Equidata receive | Equidata, Inc. | árbitros |
| obligation-32 | The arbitrator(s) shall issue an award in writing no later than 10 calendar days | National Credit Report.com, LLC | árbitros |
| obligation-53 | Executive two (2) copies and return executed copies to: | National Credit Report.com, LLC | sin referencia |

### ¿Y como segunda opinión?

En Ediets y Bellicum nadie ha revisado las partes. Donde la mejor regla y la extracción no
coinciden hay un candidato a error de una de las dos. La pregunta es si esa lista sirve para
revisar la extracción o si casi todo son falsas alarmas. En SteelVault, donde la extracción
acierta todas las partes, cada desacuerdo es una falsa alarma.

In [8]:
for name in ("SteelVault", "Ediets", "Bellicum"):
    c, res = contracts[name], t1[name]
    pred = res["preds"]["sujeto"]["0"]
    truth = {sid: p for sid, _, p in res["queries"]}
    disagree = [sid for sid in truth if pred[sid] is not None and pred[sid] != truth[sid]]
    display(Markdown(f"**{name}**: {len(disagree)} desacuerdos de {len(truth)} ({len(disagree) / len(truth):.0%}). Los cinco primeros:"))
    display(lp.table([
        {"enunciado": s, "tipo": lp.KIND_ES[c.statements[s]["kind"]], "extracción": c.parties[truth[s]]["name"], "regla": c.parties[pred[s]]["name"], "texto": lp.statement_text(c.statements[s])[:90]}
        for s in disagree[:5]
    ]))

**SteelVault**: 7 desacuerdos de 47 (15%). Los cinco primeros:

| enunciado | tipo | extracción | regla | texto |
|---|---|---|---|---|
| obligation-6 | obligación | National Credit Report.com, LLC | Equidata, Inc. | Said amounts charged to Equidata will be billed separately to Marketing Affiliate and are  |
| obligation-8 | obligación | Equidata, Inc. | National Credit Report.com, LLC | Notice will be given to Marketing Affiliate in writing no less than 30 days prior to such  |
| obligation-12 | obligación | Equidata, Inc. | National Credit Report.com, LLC | the Marketing Affiliate will be billed for those items in accordance with Equidata standar |
| obligation-45 | obligación | Equidata, Inc. | National Credit Report.com, LLC | it has the ability to provide said Services in the manner described herein. |
| right-10 | derecho | Equidata, Inc. | National Credit Report.com, LLC | Upon Marketing Affiliate’s default in payment or other breach of this Agreement, Equidata  |

**Ediets**: 37 desacuerdos de 152 (24%). Los cinco primeros:

| enunciado | tipo | extracción | regla | texto |
|---|---|---|---|---|
| obligation-29 | obligación | WOMEN.COM NETWORKS, INC. | EDIETS.COM, INC. | Should eDiets elect to terminate this Agreement, Women.com shall reimburse eDiets within t |
| obligation-33 | obligación | WOMEN.COM NETWORKS, INC. | EDIETS.COM, INC. | it shall give eDiets written notice of its intention, describing the terms and conditions  |
| obligation-36 | obligación | WOMEN.COM NETWORKS, INC. | EDIETS.COM, INC. | Such reports shall contain the number of impressions and other metrics attributable to eac |
| obligation-44 | obligación | EDIETS.COM, INC. | WOMEN.COM NETWORKS, INC. | Women.com Marks by eDiets, including all goodwill arising therefrom shall inure solely to  |
| obligation-46 | obligación | WOMEN.COM NETWORKS, INC. | EDIETS.COM, INC. | All uses of eDiets Marks by Women.com, including all goodwill arising therefrom shall inur |

**Bellicum**: 90 desacuerdos de 370 (24%). Los cinco primeros:

| enunciado | tipo | extracción | regla | texto |
|---|---|---|---|---|
| obligation-14 | obligación | Bellicum Pharmaceuticals, Inc. | Miltenyi Biotec GmbH | shall cause its Subcontractors and Licensees not to use the Miltenyi Products and/or any c |
| obligation-20 | obligación | Bellicum Pharmaceuticals, Inc. | Miltenyi Biotec GmbH | Use of the Miltenyi Products by Bellicum, its Licensees or Subcontractors shall be supervi |
| obligation-21 | obligación | Bellicum Pharmaceuticals, Inc. | Miltenyi Biotec GmbH | will cause its Licensees and Subcontractors not to, without express prior written consent  |
| obligation-27 | obligación | Miltenyi Biotec GmbH | Bellicum Pharmaceuticals, Inc. | upon reasonable prior written notice to Bellicum |
| obligation-41 | obligación | Miltenyi Biotec GmbH | Bellicum Pharmaceuticals, Inc. | unless such issues solely relate to Bellicum’s Permitted Use of the relevant Miltenyi Prod |

En SteelVault la regla discrepa en 7 de 47 enunciados y las 7 son falsas alarmas. En Ediets y
Bellicum discrepa en uno de cada cuatro. Una revisión con una falsa alarma cada siete enunciados
no ahorra trabajo: como segunda opinión, T1 tampoco sirve.

## 5. T3 — relaciones entre cláusulas

**Protocolo.** Igual que T1, pero las consultas son las aristas `depends_on`, `modifies` y
`references`, y la respuesta se busca entre todas las cláusulas del contrato. Rangos
**filtrados**: si una consulta tiene varios destinos verdaderos, los otros no cuentan como
error. MRR y Hits@1, @3 y @10.

**Los gemelos van juntos.** Una disposición bilateral lleva su relación dos veces, una por
gemelo: en Ediets son 42 de las 76 aristas, en Bellicum 46 de 213. Si un gemelo cae en
entrenamiento y el otro en prueba, el modelo no tiene que deducir nada: copia. Los dos van
siempre al mismo pliegue; al final de la sección se mide cuánto cambia si no.

**Líneas base:**

- *destino más frecuente*: la cláusula que más recibe ese tipo de arista en lo conocido;
- *cercanía en el árbol*: su propia cláusula, luego la madre y las hermanas, y así;
- *número citado→cercanía*: las cláusulas que el enunciado o su párrafo citan por número
  («Section 3.2(c)», «Sections 13.1 and 13.2») van primero; detrás, por cercanía;
- *texto*: similitud TF-IDF y MiniLM entre el enunciado y el texto de cada cláusula.

Y dos combinaciones: *número citado→TransE* y *número citado→NBFNet* ponen primero lo citado y
detrás el orden del modelo de grafo. Responden a la pregunta útil: donde el texto no cita nada,
¿ordena la estructura mejor que la simple cercanía en el árbol?

In [9]:
t3 = {}
for name, repeats in (("SteelVault", 3), ("Ediets", 1), ("Bellicum", 1)):
    c = contracts[name]
    t3[name] = lp.cached(
        f"T3:{name}",
        lambda c=c, repeats=repeats: lp.run_t3(c, KGE, repeats=repeats, k=5, nbfnet_epochs=20),
        inputs={"kg": kg_prints[name], "kge": KGE, "repeats": repeats, "k": 5, "nbfnet_epochs": 20, "regex": lp.SECTION.pattern},
        depends=lp.T3_CODE,
        recompute=RECOMPUTE,
    )
for name in ("Ediets", "Bellicum"):
    display(Markdown(f"**{name}** — {len(t3[name]['queries'])} aristas"))
    display(lp.table(lp.t3_summary(t3[name]), ["método", "MRR", "peor", "mejor", "Hits@1", "Hits@3", "Hits@10"]))

**Ediets** — 76 aristas

| método | MRR | peor | mejor | Hits@1 | Hits@3 | Hits@10 |
|---|---|---|---|---|---|---|
| número citado→TransE | 0.823 | 0.823 | 0.823 | 0.737 | 0.868 | 0.961 |
| número citado→NBFNet | 0.818 | 0.818 | 0.818 | 0.737 | 0.882 | 0.921 |
| número citado→cercanía | 0.742 | 0.742 | 0.742 | 0.671 | 0.750 | 0.855 |
| número citado→MiniLM | 0.711 | 0.711 | 0.711 | 0.605 | 0.776 | 0.842 |
| TransE | 0.599 | 0.599 | 0.599 | 0.474 | 0.632 | 0.908 |
| NBFNet | 0.579 | 0.579 | 0.579 | 0.487 | 0.645 | 0.776 |
| CompGCN | 0.510 | 0.510 | 0.510 | 0.382 | 0.592 | 0.737 |
| RotatE | 0.483 | 0.483 | 0.483 | 0.382 | 0.513 | 0.711 |
| cercanía en el árbol | 0.476 | 0.476 | 0.476 | 0.303 | 0.500 | 0.816 |
| texto: TF-IDF | 0.445 | 0.445 | 0.445 | 0.316 | 0.539 | 0.711 |
| DistMult | 0.413 | 0.413 | 0.413 | 0.303 | 0.461 | 0.592 |
| texto: MiniLM | 0.357 | 0.357 | 0.357 | 0.237 | 0.382 | 0.579 |
| RGCN | 0.348 | 0.348 | 0.348 | 0.237 | 0.382 | 0.566 |
| ComplEx | 0.274 | 0.274 | 0.274 | 0.171 | 0.289 | 0.408 |
| destino más frecuente | 0.199 | 0.199 | 0.199 | 0.066 | 0.237 | 0.474 |
| TransR | 0.128 | 0.128 | 0.128 | 0.039 | 0.118 | 0.355 |
| TransH | 0.060 | 0.060 | 0.060 | 0.000 | 0.039 | 0.145 |

**Bellicum** — 213 aristas

| método | MRR | peor | mejor | Hits@1 | Hits@3 | Hits@10 |
|---|---|---|---|---|---|---|
| número citado→NBFNet | 0.797 | 0.797 | 0.797 | 0.685 | 0.901 | 0.958 |
| número citado→cercanía | 0.741 | 0.741 | 0.741 | 0.629 | 0.803 | 0.930 |
| número citado→TransE | 0.718 | 0.718 | 0.718 | 0.606 | 0.812 | 0.887 |
| número citado→MiniLM | 0.692 | 0.692 | 0.692 | 0.592 | 0.765 | 0.854 |
| NBFNet | 0.526 | 0.526 | 0.526 | 0.441 | 0.559 | 0.700 |
| RotatE | 0.489 | 0.489 | 0.489 | 0.371 | 0.554 | 0.700 |
| texto: TF-IDF | 0.449 | 0.449 | 0.449 | 0.296 | 0.521 | 0.737 |
| cercanía en el árbol | 0.441 | 0.441 | 0.441 | 0.319 | 0.408 | 0.746 |
| DistMult | 0.424 | 0.424 | 0.424 | 0.305 | 0.474 | 0.634 |
| TransE | 0.368 | 0.368 | 0.368 | 0.254 | 0.394 | 0.606 |
| CompGCN | 0.345 | 0.345 | 0.345 | 0.225 | 0.394 | 0.587 |
| texto: MiniLM | 0.328 | 0.328 | 0.328 | 0.207 | 0.380 | 0.563 |
| ComplEx | 0.192 | 0.192 | 0.192 | 0.108 | 0.202 | 0.319 |
| destino más frecuente | 0.138 | 0.138 | 0.138 | 0.075 | 0.089 | 0.300 |
| RGCN | 0.102 | 0.102 | 0.102 | 0.038 | 0.070 | 0.221 |
| TransR | 0.082 | 0.082 | 0.082 | 0.023 | 0.075 | 0.164 |
| TransH | 0.073 | 0.073 | 0.073 | 0.014 | 0.056 | 0.160 |

In [10]:
rows = []
for name in ("Ediets", "Bellicum"):
    for relation in lp.CLAUSE_RELATIONS:
        summary = {r["método"]: r for r in lp.t3_summary(t3[name], relation)}
        best = max(summary.values(), key=lambda r: r["MRR"])
        rows.append({"contrato": name, "relación": relation, "n": best["n"],
                     "número citado→cercanía": summary["número citado→cercanía"]["MRR"],
                     "número citado→TransE": summary["número citado→TransE"]["MRR"],
                     "número citado→NBFNet": summary["número citado→NBFNet"]["MRR"],
                     "mejor modelo de grafo": max((r for m, r in summary.items() if m in KGE or m == "NBFNet"), key=lambda r: r["MRR"])["método"],
                     "su MRR": max(r["MRR"] for m, r in summary.items() if m in KGE or m == "NBFNet")})
lp.table(rows)

| contrato | relación | n | número citado→cercanía | número citado→TransE | número citado→NBFNet | mejor modelo de grafo | su MRR |
|---|---|---|---|---|---|---|---|
| Ediets | depends_on | 13 | 0.888 | 0.896 | 0.892 | NBFNet | 0.454 |
| Ediets | modifies | 43 | 0.669 | 0.803 | 0.818 | TransE | 0.720 |
| Ediets | references | 20 | 0.802 | 0.819 | 0.771 | TransE | 0.522 |
| Bellicum | depends_on | 52 | 0.875 | 0.863 | 0.875 | NBFNet | 0.523 |
| Bellicum | modifies | 101 | 0.636 | 0.584 | 0.743 | NBFNet | 0.658 |
| Bellicum | references | 60 | 0.803 | 0.819 | 0.819 | RotatE | 0.418 |

¿Es real la mejora de las combinaciones? Ediets y Bellicum tienen una sola repetición, así que el
rango entre repeticiones no lo dice. Se compara arista por arista con la regla sola: diferencia de
MRR e intervalo *bootstrap* al 95%, remuestreando aristas.

In [11]:
rows = []
for name in ("Ediets", "Bellicum"):
    ranks = t3[name]["ranks"]
    for method in ("número citado→NBFNet", "número citado→TransE"):
        d, lo, hi = lp.paired_mrr_difference(ranks[method]["0"], ranks["número citado→cercanía"]["0"])
        rows.append({"contrato": name, "método": method, "MRR − regla": d, "IC95 bajo": lo, "IC95 alto": hi})
lp.table(rows)

| contrato | método | MRR − regla | IC95 bajo | IC95 alto |
|---|---|---|---|---|
| Ediets | número citado→NBFNet | 0.077 | 0.006 | 0.153 |
| Ediets | número citado→TransE | 0.081 | 0.035 | 0.133 |
| Bellicum | número citado→NBFNet | 0.055 | 0.029 | 0.085 |
| Bellicum | número citado→TransE | -0.023 | -0.059 | 0.013 |

En SteelVault solo hay siete aristas, y cuatro de ellas apuntan a la misma cláusula (§7.1, la
duración del contrato). Lo que ahí acierte un modelo es, sobre todo, haber visto esa frecuencia:

In [12]:
lp.table(lp.t3_summary(t3["SteelVault"]), ["método", "MRR", "peor", "mejor", "Hits@1", "Hits@3", "Hits@10"])

| método | MRR | peor | mejor | Hits@1 | Hits@3 | Hits@10 |
|---|---|---|---|---|---|---|
| TransE | 0.749 | 0.742 | 0.759 | 0.714 | 0.714 | 0.905 |
| número citado→TransE | 0.749 | 0.742 | 0.759 | 0.714 | 0.714 | 0.905 |
| destino más frecuente | 0.602 | 0.602 | 0.602 | 0.571 | 0.571 | 0.571 |
| RotatE | 0.526 | 0.493 | 0.584 | 0.333 | 0.714 | 0.810 |
| NBFNet | 0.506 | 0.369 | 0.659 | 0.333 | 0.619 | 0.714 |
| número citado→NBFNet | 0.506 | 0.369 | 0.659 | 0.333 | 0.619 | 0.714 |
| CompGCN | 0.476 | 0.360 | 0.611 | 0.381 | 0.476 | 0.667 |
| DistMult | 0.464 | 0.325 | 0.542 | 0.333 | 0.524 | 0.619 |
| cercanía en el árbol | 0.448 | 0.448 | 0.448 | 0.286 | 0.286 | 0.857 |
| número citado→cercanía | 0.448 | 0.448 | 0.448 | 0.286 | 0.286 | 0.857 |
| RGCN | 0.426 | 0.347 | 0.493 | 0.286 | 0.524 | 0.619 |
| TransR | 0.411 | 0.240 | 0.499 | 0.286 | 0.381 | 0.667 |
| TransH | 0.312 | 0.109 | 0.490 | 0.190 | 0.286 | 0.524 |
| texto: TF-IDF | 0.270 | 0.270 | 0.270 | 0.143 | 0.286 | 0.571 |
| texto: MiniLM | 0.266 | 0.266 | 0.266 | 0.143 | 0.143 | 0.714 |
| número citado→MiniLM | 0.266 | 0.266 | 0.266 | 0.143 | 0.143 | 0.714 |
| ComplEx | 0.068 | 0.051 | 0.093 | 0.000 | 0.000 | 0.190 |

**Si los gemelos se reparten al azar.** El mismo experimento en Ediets, con los dos mejores
modelos de grafo y sin agrupar los gemelos:

In [13]:
E = contracts["Ediets"]
leaky = lp.cached(
    "T3:Ediets:gemelos repartidos",
    lambda: lp.run_t3(E, {"TransE": KGE["TransE"]}, repeats=1, k=5, nbfnet_epochs=20, group_twins=False),
    inputs={"kg": kg_prints["Ediets"], "kge": KGE["TransE"], "repeats": 1, "k": 5, "nbfnet_epochs": 20, "regex": lp.SECTION.pattern, "twins": "repartidos"},
    depends=lp.T3_CODE,
    recompute=RECOMPUTE,
)
fair = {r["método"]: r["MRR"] for r in lp.t3_summary(t3["Ediets"])}
lp.table([
    {"método": r["método"], "MRR con gemelos juntos": fair[r["método"]], "MRR con gemelos repartidos": r["MRR"]}
    for r in lp.t3_summary(leaky) if r["método"] in ("TransE", "NBFNet", "número citado→cercanía")
])

| método | MRR con gemelos juntos | MRR con gemelos repartidos |
|---|---|---|
| número citado→cercanía | 0.742 | 0.742 |
| NBFNet | 0.579 | 0.734 |
| TransE | 0.599 | 0.731 |

**Lectura.** Sola, la regla del número citado supera a todos los modelos de grafo: MRR 0,742 en
Ediets y 0,741 en Bellicum, frente a 0,599 y 0,526 del mejor. Pero puesta delante de NBFNet,
para que el modelo ordene solo lo que el texto no cita, la combinación mejora a la regla con
cercanía: +0,077 en Ediets y +0,055 en Bellicum, con intervalos que no tocan el cero. La ganancia
está casi entera en `modifies` —«notwithstanding the foregoing», «this Section shall not…»—, la
relación que menos se nombra con números. Con TransE detrás, la mejora no se repite en Bellicum.

Dos salvedades. Las aristas de Ediets y Bellicum las produjo la extracción y nadie las ha
revisado: lo que NBFNet aprende es la costumbre del extractor, no necesariamente la verdad. Y con
los gemelos repartidos al azar, los modelos de grafo empatan con la regla (0,73 frente a 0,74)
solo porque copian la arista del gemelo.

## 6. T2 — de la consecuencia al deber incumplido

### La anotación que hace falta — **propuesta, pendiente de revisión**

El grafo no tiene ni un ejemplo de esta relación, así que se anota a mano sobre la referencia de
SteelVault: para cada condición, si su disparador es un incumplimiento y, si lo es, qué deber
se incumple. Las 20 condiciones de la referencia (17 obligatorias y 3 opcionales) caen en cuatro
clases:

- **específica**: el disparador nombra el deber, o lo deja claro («by reason of non payment»);
- **genérica**: cualquier incumplimiento de una parte («breach of this Agreement by Marketing
  Affiliate»). No se puede ordenar —todos sus deberes son correctos— y no entra en la medida,
  pero sí en los caminos de exposición;
- **no es incumplimiento**: un hecho pactado, una modalidad, un tercero, la terminación;
- **dudosa**: se deja fuera de la medida hasta que se decida.

Los deberes se anotan con los ids de la referencia (`S06`, `W4`…), no con los del grafo: así la
anotación sobrevive a una nueva extracción. «Deberes de pago» son todos los del Affiliate que
mandan pagar algo principal; los recargos, intereses y honorarios no, porque son consecuencias.

**Esta tabla es una propuesta para revisar antes de dar por buenas las cifras de T2.** Si se
aprueba, su sitio es [`gold_steelvault.ipynb`](gold_steelvault.ipynb), dentro de la referencia.

In [14]:
PAGO = ["S06", "S08", "S09", "S13", "S14", "S15", "S23", "S46"]
# (condición, consecuencia que abre, clase, deberes incumplidos, por qué)
PROPOSAL = [
    ("C01", "S03", "no es incumplimiento", [], "«express written permission»: una excepción a la prohibición, no un incumplimiento"),
    ("C02", "S10", "específica", PAGO, "«by reason of non payment»: el impago de lo que el Affiliate debe"),
    ("C03", "S13", "no es incumplimiento", [], "«upon a signed agreement of project scope»: un hecho pactado"),
    ("C04", "S15", "no es incumplimiento", [], "«when paying by credit card»: una modalidad de pago"),
    ("C05", "S19", "específica", PAGO, "«referred to an attorney for collection»: la remisión al abogado presupone el impago"),
    ("C06", "S17", "específica", PAGO, "«on the unpaid, past-due amount»: el recargo cae sobre lo impagado"),
    ("C07", "S20a", "no es incumplimiento", [], "«disputed charge»: retener una factura con aviso de disputa está previsto; abre el arbitraje"),
    ("C08", "S27a", "no es incumplimiento", [], "no resolver la disputa en 10 días: ningún deber obliga a ponerse de acuerdo"),
    ("C09", "S28a", "dudosa", ["S27a", "S27b"], "no acordar el árbitro es no cumplir el deber conjunto de §2, pero sin culpa de nadie: es un mecanismo de respaldo"),
    ("C10", "S44", "específica", PAGO, "«default in payment or other breach»: el impago, y además cualquier incumplimiento (la parte genérica no se mide)"),
    ("C11", "S45", "no es incumplimiento", [], "«upon termination for any reason»"),
    ("C12", "S47", "no es incumplimiento", [], "«at time of Agreement termination»"),
    ("C13", "S48", "específica", ["S47"], "«if data is not voluntarily removed»: el deber de retirar los datos al terminar"),
    ("C14", "S50", "no es incumplimiento", [], "las agencias de crédito dejan de dar servicio: un hecho de un tercero"),
    ("C15", "S50", "no es incumplimiento", [], "las agencias avisan a Equidata: un hecho de un tercero"),
    ("C16", "S51a", "específica", ["W4"], "«breach of any warranties… or failure to provide Services»: Equidata indemniza por lo suyo, su garantía de §8.4"),
    ("C16", "S51b", "específica", ["W1", "W2", "W3", "W1p"], "el mismo disparador: el Affiliate indemniza por sus garantías de §8.1–8.3"),
    ("C17", "S57", "genérica", [], "«breach of this Agreement by Marketing Affiliate»: cualquier deber del Affiliate"),
    ("C18", "S43a", "dudosa", [], "«with cause»: suele querer decir un incumplimiento de la otra parte, pero no lo dice"),
    ("C19", "S49", "específica", ["S47"], "el mismo disparador que C13: el Affiliate paga el coste de forzar la retirada"),
    ("C20", "S16", "específica", PAGO, "«any unpaid balance past due on any invoice»: el avalista responde del impago"),
]
gold_text = {g["id"]: g["text"] for g in gold["statements"]}
lp.table([
    {"condición": c, "consecuencia": f"{s}: {gold_text[s][:60]}", "clase": k,
     "deber incumplido": ", ".join(b) if b != PAGO else "deberes de pago (" + ", ".join(PAGO) + ")", "por qué": why}
    for c, s, k, b, why in PROPOSAL
])

| condición | consecuencia | clase | deber incumplido | por qué |
|---|---|---|---|---|
| C01 | S03: Marketing Affiliate may not otherwise offer for sale, market | no es incumplimiento |  | «express written permission»: una excepción a la prohibición, no un incumplimiento |
| C02 | S10: Marketing Affiliate agrees to reimburse Equidata all costs o | específica | deberes de pago (S06, S08, S09, S13, S14, S15, S23, S46) | «by reason of non payment»: el impago de lo que el Affiliate debe |
| C03 | S13: A development fee, yet To Be Determined and outlined in Exhi | no es incumplimiento |  | «upon a signed agreement of project scope»: un hecho pactado |
| C04 | S15: When paying by credit card, the Marketing Affiliate agrees t | no es incumplimiento |  | «when paying by credit card»: una modalidad de pago |
| C05 | S19: the Marketing Affiliate agrees to pay 25% attorney’s fees pl | específica | deberes de pago (S06, S08, S09, S13, S14, S15, S23, S46) | «referred to an attorney for collection»: la remisión al abogado presupone el impago |
| C06 | S17: The Marketing Affiliate agrees to pay a late charge of 1 l/2 | específica | deberes de pago (S06, S08, S09, S13, S14, S15, S23, S46) | «on the unpaid, past-due amount»: el recargo cae sobre lo impagado |
| C07 | S20a: Equidata or Marketing Affiliate may choose arbitration | no es incumplimiento |  | «disputed charge»: retener una factura con aviso de disputa está previsto; abre el arbitraje |
| C08 | S27a: the parties shall jointly select an arbitrator | no es incumplimiento |  | no resolver la disputa en 10 días: ningún deber obliga a ponerse de acuerdo |
| C09 | S28a: each party will select an arbitrator of their choosing | dudosa | S27a, S27b | no acordar el árbitro es no cumplir el deber conjunto de §2, pero sin culpa de nadie: es un mecanismo de respaldo |
| C10 | S44: Upon Marketing Affiliate’s default in payment or other breac | específica | deberes de pago (S06, S08, S09, S13, S14, S15, S23, S46) | «default in payment or other breach»: el impago, y además cualquier incumplimiento (la parte genérica no se mide) |
| C11 | S45: Upon termination for any reason, Equidata reserves the right | no es incumplimiento |  | «upon termination for any reason» |
| C12 | S47: At time of Agreement termination, Marketing Affiliate shall  | no es incumplimiento |  | «at time of Agreement termination» |
| C13 | S48: if data is not voluntarily removed, Equidata reserves the ri | específica | S47 | «if data is not voluntarily removed»: el deber de retirar los datos al terminar |
| C14 | S50: Equidata reserves the right to terminate this Agreement imme | no es incumplimiento |  | las agencias de crédito dejan de dar servicio: un hecho de un tercero |
| C15 | S50: Equidata reserves the right to terminate this Agreement imme | no es incumplimiento |  | las agencias avisan a Equidata: un hecho de un tercero |
| C16 | S51a: Equidata and Marketing Affiliate each hereby agree to defend | específica | W4 | «breach of any warranties… or failure to provide Services»: Equidata indemniza por lo suyo, su garantía de §8.4 |
| C16 | S51b: Equidata and Marketing Affiliate each hereby agree to defend | específica | W1, W2, W3, W1p | el mismo disparador: el Affiliate indemniza por sus garantías de §8.1–8.3 |
| C17 | S57: MARKETING AFFILIATE AGREES TO DEFEND, INDEMNIFY AND HOLD HAR | genérica |  | «breach of this Agreement by Marketing Affiliate»: cualquier deber del Affiliate |
| C18 | S43a: This Agreement may be terminated by either party with cause  | dudosa |  | «with cause»: suele querer decir un incumplimiento de la otra parte, pero no lo dice |
| C19 | S49: Marketing Affiliate agrees to be liable for the cost of such | específica | S47 | el mismo disparador que C13: el Affiliate paga el coste de forzar la retirada |
| C20 | S16: this Agreement will serve as a personal guaranty by the unde | específica | deberes de pago (S06, S08, S09, S13, S14, S15, S23, S46) | «any unpaid balance past due on any invoice»: el avalista responde del impago |

### Protocolo

Cada fila *específica* es una consulta: dada la consecuencia —y el texto de la condición que la
abre—, ordenar todos los deberes del contrato (obligaciones, garantías y prohibiciones) y ver
dónde cae el primero correcto. No hay nada que ocultar ni pliegues: ningún método ha visto nunca
esta relación.

- *orden del documento*: el deber escrito más cerca, y a igual distancia el anterior;
- *texto*: similitud TF-IDF o MiniLM entre «disparador + consecuencia» y cada deber;
- *estructura*: cercanía entre los vectores que un modelo de grafo aprende del grafo entero. Es
  lo único que puede aportar: sin ejemplos de la relación, no hay nada que entrenar, y NBFNet,
  que necesita consultas de entrenamiento, no se puede usar;
- *mismo beneficiario → …*: primero los deberes que protegen a la misma parte a la que sirve la
  consecuencia —el remedio va a quien era acreedor del deber incumplido—, y dentro, el orden del
  método. Es una regla que solo usa campos del grafo.

Con tan pocas consultas el intervalo es ancho: se da el del *bootstrap* al 95%.

In [15]:
import collections
import re

t2_q = lp.t2_queries(SV, gold, PROPOSAL)
e2i, r2i = lp.entity_ids(SV)
structure = {}
for m in ("TransE", "RGCN", "CompGCN"):
    for seed in (0, 1, 2):  # el vector de cada entidad depende de la semilla: tres, para ver cuánto
        structure[f"{m} #{seed}"] = (lp.kge_entity_vectors(lp.train_kge(m, SV.triples, e2i, r2i, seed=seed, **KGE[m])), e2i)
t2 = lp.run_t2(SV, t2_q, structure)


def by_method(rows):
    # Los métodos de estructura salen una vez por semilla: media, peor y mejor.
    groups = collections.defaultdict(list)
    for r in rows:
        groups[re.sub(r" #\d+$", "", r["método"])].append(r)
    out = []
    for name, rs in groups.items():
        mrr = [r["MRR"] for r in rs]
        out.append({"método": name, "MRR": float(np.mean(mrr)), "peor": min(mrr), "mejor": max(mrr),
                    **{k: float(np.mean([r[k] for r in rs])) for k in ("Hits@1", "Hits@3", "Hits@10")},
                    "IC95 bajo": rs[0]["IC95 bajo"] if len(rs) == 1 else float("nan"),
                    "IC95 alto": rs[0]["IC95 alto"] if len(rs) == 1 else float("nan")})
    return sorted(out, key=lambda r: -r["MRR"])


display(Markdown(f"{len(t2_q)} consultas, entre {t2['rows'][0]['candidatos']} deberes candidatos. Intervalo *bootstrap* al 95% donde el método es determinista."))
lp.table(by_method(lp.t2_summary(t2)), ["método", "MRR", "peor", "mejor", "Hits@1", "Hits@3", "Hits@10", "IC95 bajo", "IC95 alto"])

9 consultas, entre 71 deberes candidatos. Intervalo *bootstrap* al 95% donde el método es determinista.

| método | MRR | peor | mejor | Hits@1 | Hits@3 | Hits@10 | IC95 bajo | IC95 alto |
|---|---|---|---|---|---|---|---|---|
| mismo beneficiario → estructura: CompGCN | 0.712 | 0.630 | 0.800 | 0.556 | 0.889 | 0.926 | — | — |
| mismo beneficiario → estructura: RGCN | 0.695 | 0.514 | 0.819 | 0.556 | 0.778 | 0.963 | — | — |
| mismo beneficiario → estructura: TransE | 0.665 | 0.614 | 0.694 | 0.519 | 0.778 | 0.963 | — | — |
| estructura: CompGCN | 0.653 | 0.559 | 0.713 | 0.481 | 0.815 | 0.926 | — | — |
| estructura: RGCN | 0.629 | 0.492 | 0.744 | 0.481 | 0.704 | 0.926 | — | — |
| estructura: TransE | 0.612 | 0.596 | 0.630 | 0.444 | 0.704 | 0.889 | — | — |
| mismo beneficiario → orden del documento | 0.550 | 0.550 | 0.550 | 0.222 | 0.778 | 1.000 | 0.372 | 0.748 |
| orden del documento | 0.480 | 0.480 | 0.480 | 0.222 | 0.556 | 0.778 | 0.280 | 0.706 |
| mismo beneficiario → texto: MiniLM | 0.461 | 0.461 | 0.461 | 0.222 | 0.556 | 1.000 | 0.270 | 0.676 |
| texto: MiniLM | 0.397 | 0.397 | 0.397 | 0.222 | 0.333 | 1.000 | 0.189 | 0.636 |
| mismo beneficiario → texto: TF-IDF | 0.357 | 0.357 | 0.357 | 0.222 | 0.222 | 1.000 | 0.162 | 0.629 |
| al azar, mismo beneficiario (esperado) | 0.288 | 0.288 | 0.288 | 0.123 | 0.322 | 0.701 | — | — |
| texto: TF-IDF | 0.264 | 0.264 | 0.264 | 0.111 | 0.222 | 0.889 | 0.133 | 0.462 |
| al azar (esperado) | 0.190 | 0.190 | 0.190 | 0.067 | 0.187 | 0.484 | — | — |

El rango de cada consulta, para ver dónde falla cada método:

In [16]:
show = ["mismo beneficiario → orden del documento", "mismo beneficiario → texto: MiniLM", "mismo beneficiario → estructura: TransE #0", "mismo beneficiario → estructura: CompGCN #0"]
lp.table([{"condición": r["condición"], "consecuencia": r["consecuencia"], **{m: r[m] for m in show}} for r in t2["rows"]], digits=1)

| condición | consecuencia | mismo beneficiario → orden del documento | mismo beneficiario → texto: MiniLM | mismo beneficiario → estructura: TransE #0 | mismo beneficiario → estructura: CompGCN #0 |
|---|---|---|---|---|---|
| C02 | obligation-7 | 2.5 | 1.0 | 4.0 | 1.0 |
| C05 | obligation-16 | 3.0 | 4.0 | 1.0 | 1.0 |
| C06 | obligation-14 | 2.0 | 2.0 | 1.0 | 1.0 |
| C10 | right-10 | 1.0 | 3.0 | 1.0 | 1.0 |
| C13 | right-12 | 1.5 | 1.0 | 2.0 | 1.0 |
| C16 | obligation-46 | 5.5 | 2.0 | 1.0 | 6.0 |
| C16 | obligation-47 | 5.0 | 5.0 | 12.0 | 29.0 |
| C19 | obligation-39 | 1.0 | 5.0 | 1.0 | 1.0 |
| C20 | obligation-13 | 1.5 | 6.0 | 3.0 | 1.0 |

**Lectura — provisional mientras la anotación no esté revisada.** Con el filtro de beneficiario,
la cercanía entre los vectores de un modelo de grafo pone el deber correcto primero en más
consultas que el orden del documento o el texto: CompGCN da 0,71 de MRR de media entre semillas
(0,63–0,80), frente a 0,55 del orden del documento. Con nueve consultas no se puede dar por
ganador a ninguno: el intervalo del orden del documento va de 0,37 a 0,75. Todos fallan donde la
consecuencia y el deber están en cláusulas distintas y no comparten términos, como la
indemnización de §9 por las garantías de §8.

El filtro de beneficiario es una regla del esquema, no un modelo, y mejora a todos los métodos
(de 0,05 a 0,09 de MRR). Con él, el orden del documento pone siempre el deber correcto entre los
diez primeros: es una lista corta que se revisa a mano.

### A nivel de cláusula — lo que cabe en el esquema actual

Una arista de un enunciado a otro no existe en el esquema. Una de un enunciado a una cláusula,
sí: `depends_on` ya sale de enunciados y llega a cláusulas. La versión de T2 que cabe sin
tocar nada es, por tanto, «la consecuencia depende de la cláusula donde está el deber». Se
puntúa cada cláusula por su mejor deber:

In [17]:
lp.table(by_method(lp.t2_summary(t2, prefix="[cláusula] ")), ["método", "MRR", "peor", "mejor", "Hits@1", "Hits@3", "Hits@10", "IC95 bajo", "IC95 alto"])

| método | MRR | peor | mejor | Hits@1 | Hits@3 | Hits@10 | IC95 bajo | IC95 alto |
|---|---|---|---|---|---|---|---|---|
| mismo beneficiario → estructura: TransE | 0.904 | 0.901 | 0.907 | 0.889 | 0.889 | 1.000 | — | — |
| estructura: TransE | 0.863 | 0.843 | 0.901 | 0.815 | 0.889 | 0.963 | — | — |
| mismo beneficiario → orden del documento | 0.843 | 0.843 | 0.843 | 0.778 | 0.889 | 1.000 | 0.667 | 1.000 |
| mismo beneficiario → estructura: CompGCN | 0.842 | 0.813 | 0.870 | 0.778 | 0.889 | 0.926 | — | — |
| estructura: CompGCN | 0.821 | 0.807 | 0.841 | 0.741 | 0.889 | 0.926 | — | — |
| orden del documento | 0.819 | 0.819 | 0.819 | 0.778 | 0.778 | 1.000 | 0.630 | 1.000 |
| mismo beneficiario → estructura: RGCN | 0.760 | 0.591 | 0.861 | 0.667 | 0.778 | 0.963 | — | — |
| estructura: RGCN | 0.723 | 0.571 | 0.856 | 0.630 | 0.741 | 0.963 | — | — |
| mismo beneficiario → texto: TF-IDF | 0.637 | 0.637 | 0.637 | 0.444 | 0.778 | 1.000 | 0.407 | 0.856 |
| mismo beneficiario → texto: MiniLM | 0.624 | 0.624 | 0.624 | 0.444 | 0.778 | 1.000 | 0.396 | 0.843 |
| texto: MiniLM | 0.588 | 0.588 | 0.588 | 0.444 | 0.667 | 1.000 | 0.338 | 0.833 |
| texto: TF-IDF | 0.555 | 0.555 | 0.555 | 0.333 | 0.778 | 1.000 | 0.342 | 0.781 |

A nivel de cláusula, lo que cabría hoy en el grafo, el orden del documento ya acierta la
cláusula en 7 de 9 consultas, y TransE con el filtro de beneficiario en 8 de 9.

### Con un modelo de lenguaje — coste estimado, sin ejecutar

La línea base que falta es preguntárselo a un LLM: la lista de deberes del contrato, la
consecuencia y su disparador, y que devuelva los tres deberes más probables. Las instrucciones y
la lista son iguales en todas las llamadas, así que van a la caché. Estimación con los precios
de [`configuracion.md`](../../docs/extraccion/configuracion.md) y el contador de tokens del
servidor:

In [18]:
import sys

sys.path.insert(0, str(lp.ROOT / "server"))
from services.llm.cost_estimator import MODEL_PRICING_USD_PER_1M, estimate_tokens

MODEL = "gpt-6.1-sol"
listing = "\n".join(f"- {d}: {lp.statement_text(SV.statements[d])}" for d in lp.duties(SV))
instructions = "Given a consequence in a contract and the condition that triggers it, return the ids of the three duties whose breach most likely triggers it, best first, as JSON."
gates = lp.gating_conditions(SV)
consequences = [sid for sid in SV.statements if gates.get(sid)]
shared = estimate_tokens(instructions + listing, MODEL)
own = [estimate_tokens(" ".join(c["trigger"] for c in gates[s]) + lp.statement_text(SV.statements[s]), MODEL) for s in consequences]
out_tokens = 300  # tres ids, una frase de motivo y el razonamiento mínimo
rates = MODEL_PRICING_USD_PER_1M[MODEL].short


def cost(n):
    first = (shared * rates.cache_write + own[0] * rates.input) / 1e6
    rest = sum(shared * rates.cached_input + o * rates.input for o in own[1:n]) / 1e6
    return first + rest + n * out_tokens * rates.output / 1e6


lp.table([
    {"qué": f"las {len(t2_q)} consultas medidas", "llamadas": len(t2_q), "tokens compartidos": shared, "coste estimado (USD)": cost(len(t2_q))},
    {"qué": "todas las consecuencias con condición de SteelVault", "llamadas": len(consequences), "tokens compartidos": shared, "coste estimado (USD)": cost(len(consequences))},
], digits=3)

| qué | llamadas | tokens compartidos | coste estimado (USD) |
|---|---|---|---|
| las 9 consultas medidas | 9 | 2488 | 0.036 |
| todas las consecuencias con condición de SteelVault | 28 | 2488 | 0.099 |

## 7. De las aristas al veredicto, sin pesos

El reparto actual suma pesos —derecho 0,3, obligación 0,7, prohibición 1,0— que nada valida, y
no usa ninguna arista del grafo más allá de las de parte. La alternativa no elige pesos:
**solo afirma lo que sería cierto con cualquier elección de pesos**, y cada afirmación lleva el
camino de aristas que la justifica.

**A quién favorece una cláusula.** Cada enunciado unilateral sirve a una parte: la que se
beneficia, al otro extremo de su arista de parte. Los gemelos son recíprocos y no inclinan nada.

- **unánime**: todos sirven a la misma parte. Cierto con cualquier peso por enunciado.
- **dominante**: una parte recibe al menos tantos derechos, tantas obligaciones a su favor y
  tantas prohibiciones a su favor como la otra, y más en alguno. Cierto con cualquier peso
  positivo por tipo: ningún slider puede darle la vuelta.
- **depende de los pesos**: lo demás. La vista no da ganador; enseña las dos listas.

**Qué exposición genera.** Un camino por cada consecuencia de un incumplimiento:

```
deber de A  ──(T2: depends_on)──  consecuencia a favor de B  ◄──is_part_of── condición (disparador)
                                          ▲
                                          └──is_part_of── valor (25 %, 30 días, 1,5 % al mes)
```

Todo sale del grafo salvo la primera arista, que es la anotada o la predicha, y se guarda fuera.
El número de caminos por parte es una cuenta verificable, no un peso: cada uno se puede abrir.

**Qué recorta qué.** Un derecho de A que `modifies` una cláusula donde A debe algo a B le quita a B
esa protección. Es la forma en que el grafo dice «la exclusión de §11 vacía la garantía de §8.4».

In [19]:
verdicts = lp.clause_verdicts(SV)
rows = []
for v in verdicts:
    share = lp.weighted_share(SV, v["cláusula"])
    total = sum(share.values())
    lead = max(share, key=share.get) if total and len(set(share.values())) > 1 else None
    rows.append({
        "§": v["ref"], "título": v["título"][:28], "veredicto": v["veredicto"], "a favor de": v["a favor de"],
        "sirve a Equidata": v["sirve a Equidata, Inc."], "sirve al Affiliate": v["sirve a National Credit Report.com, LLC"],
        "recíprocos": v["recíprocos"],
        "barra actual (Equidata)": share.get("party-1", 0) / total if total else float("nan"),
    })
lp.table(rows, digits=2)

| § | título | veredicto | a favor de | sirve a Equidata | sirve al Affiliate | recíprocos | barra actual (Equidata) |
|---|---|---|---|---|---|---|---|
| 1 | Compensation | dominante | Equidata, Inc. | obligation 12, right 1 | obligation 4 | 0 | 0.76 |
| 2 | Disputes | unánime | Equidata, Inc. | obligation 3 | — | 10 | 0.63 |
| 3 | Operational Specifications | recíproca |  | — | — | 2 | 0.50 |
| 4 | Non-solicitation of Clients | unánime | Equidata, Inc. | prohibition 2 | — | 0 | 1.00 |
| 5 | Compliance | recíproca |  | — | — | 2 | 0.50 |
| 5.1 |  | recíproca |  | — | — | 2 | 0.50 |
| 5.2 |  | recíproca |  | — | — | 2 | 0.50 |
| 5.3 |  | recíproca |  | — | — | 2 | 0.50 |
| 5.4 |  | unánime | National Credit Report.com, LLC | — | obligation 1, prohibition 1 | 2 | 0.27 |
| 5.5 |  | recíproca |  | — | — | 2 | 0.50 |
| 6 | Audit | depende de los pesos |  | right 2 | obligation 1 | 0 | 0.46 |
| 7.2 |  | unánime | Equidata, Inc. | obligation 1, right 2 | — | 2 | 0.84 |
| 7.3 |  | unánime | Equidata, Inc. | obligation 2, right 1 | — | 0 | 1.00 |
| 7.4 |  | unánime | Equidata, Inc. | right 1 | — | 0 | 1.00 |
| 8.1 |  | unánime | Equidata, Inc. | obligation 2 | — | 0 | 1.00 |
| 8.2 |  | unánime | Equidata, Inc. | obligation 1 | — | 0 | 1.00 |
| 8.3 |  | unánime | Equidata, Inc. | obligation 1 | — | 0 | 1.00 |
| 8.4 |  | unánime | National Credit Report.com, LLC | — | obligation 2 | 0 | 0.00 |
| 9 | Indemnification | recíproca |  | — | — | 2 | 0.50 |
| 10 | Proprietary Information | unánime | National Credit Report.com, LLC | — | prohibition 1 | 2 | 0.33 |
| 11 | Liability | unánime | Equidata, Inc. | obligation 1, right 3 | — | 0 | 1.00 |
| 12 | Miscellaneous | recíproca |  | — | — | 6 | 0.50 |
| — | (sin cláusula) | depende de los pesos |  | prohibition 1 | right 2 | 0 | 0.62 |

In [20]:
from collections import Counter

for name, c in contracts.items():
    counts = Counter(v["veredicto"] for v in lp.clause_verdicts(c))
    display(Markdown(f"**{name}**: " + ", ".join(f"{k} {n}" for k, n in counts.most_common())))

**SteelVault**: unánime 12, recíproca 8, depende de los pesos 2, dominante 1

**Ediets**: recíproca 23, unánime 13, depende de los pesos 7, dominante 6

**Bellicum**: depende de los pesos 28, unánime 23, recíproca 20, dominante 20

Las cláusulas que «dependen de los pesos» son justo las que la barra actual resuelve por los
pesos. Con tres juegos razonables —los de ahora, contar cada disposición igual y los de ahora al
revés—, ¿cuántas cambian de ganador?

In [21]:
for name, c in contracts.items():
    flips = lp.weight_flips(c)
    display(Markdown(f"**{name}**: {sum(r['cambia de ganador'] for r in flips)} de {len(flips)} cambian de ganador"))
lp.table(lp.weight_flips(SV))

**SteelVault**: 2 de 2 cambian de ganador

**Ediets**: 2 de 7 cambian de ganador

**Bellicum**: 11 de 28 cambian de ganador

| § | título | pesos actuales (0,3 · 0,7 · 1,0) | contar (1 · 1 · 1) | al revés (1,0 · 0,7 · 0,3) | cambia de ganador |
|---|---|---|---|---|---|
| 6 | Audit | National Credit Re 54% | Equidata, Inc. 67% | Equidata, Inc. 74% | True |
| — | (sin cláusula) | Equidata, Inc. 62% | National Credit Re 67% | National Credit Re 87% | True |

En Bellicum, un tercio de las cláusulas con enunciados (28 de 91) dependen de los pesos, y en
11 de ellas basta cambiar los pesos para que la barra cambie de ganador. §6 *Audit* de
SteelVault es el caso más claro: sus dos derechos son de Equidata, y la barra actual le da el 54%
al Affiliate porque el único deber de Equidata —pagar la auditoría— pesa 0,7 y cada derecho 0,3.

**Exposición en SteelVault**, con los vínculos de la anotación propuesta (específicos y
genéricos):

In [22]:
one = lp.match_gold_statements(SV, gold)
many = lp.match_gold_statements(SV, gold, one_to_one=False)
affiliate = next(p for p, x in SV.parties.items() if "Marketing Affiliate" in x.get("aliases", []))
links = {}
for cond, consequence, kind, breached, _ in PROPOSAL:
    if consequence not in one:
        continue
    if kind == "específica":
        links[one[consequence]] = {"duties": sorted(set().union(*(many.get(b, set()) for b in breached))), "source": "anotado"}
    elif kind == "genérica":
        links[one[consequence]] = {"duties": [], "generic": affiliate, "source": "anotado (genérico)"}
paths = lp.exposure_paths(SV, links)
display(lp.table(paths, ["expone a", "deber incumplido", "disparador", "consecuencia", "tipo", "a favor de", "cifras"]))
display(Markdown("Caminos por parte expuesta: " + ", ".join(f"{k} {n}" for k, n in Counter(p["expone a"] for p in paths).items())))

| expone a | deber incumplido | disparador | consecuencia | tipo | a favor de | cifras |
|---|---|---|---|---|---|---|
| National Credit Report.com, LLC | obligation-3 (§1); obligation-6 (§1) y 5 más (§1, §2, §7.2) | past due amounts from Marketing Affiliate by reason of non payment | obligation-7 (§1) | obligación | Equidata, Inc. |  |
| National Credit Report.com, LLC | obligation-3 (§1); obligation-6 (§1) y 5 más (§1, §2, §7.2) | the Marketing Affiliate’s account is referred to an attorney for collection | obligation-16 (§1) | obligación | Equidata, Inc. | 25 % |
| National Credit Report.com, LLC | obligation-3 (§1); obligation-6 (§1) y 5 más (§1, §2, §7.2) | the unpaid, past-due amount | obligation-14 (§1) | obligación | Equidata, Inc. | 1 l/2 % per month |
| National Credit Report.com, LLC | obligation-3 (§1); obligation-6 (§1) y 5 más (§1, §2, §7.2) | Marketing Affiliate’s default in payment or other breach of this Agreement | right-10 (§7.2) | derecho | Equidata, Inc. |  |
| National Credit Report.com, LLC | obligation-38 (§7.3) | if data is not voluntarily removed | right-12 (§7.3) | derecho | Equidata, Inc. |  |
| Equidata, Inc. | obligation-44 (§8.4); obligation-45 (§8.4) | any claims, suits, demand or actions arising from breach of any warranties under this Agreement or failure to provide Services under this Agreement | obligation-46 (§9) | obligación | National Credit Report.com, LLC |  |
| National Credit Report.com, LLC | obligation-40 (§8.1); obligation-41 (§8.1) y 2 más (§8.2, §8.3) | any claims, suits, demand or actions arising from breach of any warranties under this Agreement or failure to provide Services under this Agreement | obligation-47 (§9) | obligación | Equidata, Inc. |  |
| National Credit Report.com, LLC | cualquier deber de National Credit Report.com, LLC | ANY CLAIM, SUIT, DEMAND, OR ACTION, INCLUDING WITHOUT LIMITATION ATTORNEY FEES, ARISING FROM (A) BREACH OF THIS AGREEMENT BY MARKETING AFFILIATE, (B) THE MARKETING AFFILIATE WEB SITE, OR (C) MARKETING AFFILIATE’S BUSINESS | obligation-48 (§11) | obligación | Equidata, Inc. |  |
| National Credit Report.com, LLC | obligation-38 (§7.3) | if data is not voluntarily removed | obligation-39 (§7.3) | obligación | Equidata, Inc. |  |
| National Credit Report.com, LLC | obligation-3 (§1); obligation-6 (§1) y 5 más (§1, §2, §7.2) | any unpaid balance past due on any invoice | obligation-13 (§1) | obligación | Equidata, Inc. |  |

Caminos por parte expuesta: National Credit Report.com, LLC 9, Equidata, Inc. 1

In [23]:
lp.table(lp.limiting_paths(SV))

| derecho que recorta | § derecho | titular | cláusula recortada | deber recortado | a quien protegía |
|---|---|---|---|---|---|
| right-15 | 11 | Equidata, Inc. | 8.4 | obligation-44 | National Credit Report.com, LLC |
| right-15 | 11 | Equidata, Inc. | 8.4 | obligation-45 | National Credit Report.com, LLC |

**Con los vínculos predichos.** Lo mismo, pero el deber de cada camino es el primero que propone
la regla *mismo beneficiario → orden del documento*, en vez del anotado. Es la que se recomienda
para proponer candidatos: determinista, explicable en una frase («el deber más cercano que
protege a la misma parte») y sin semillas. La tabla cuenta cuántos caminos cambian de deudor o
de cláusula: es lo que vería mal la vista si se fiara de la predicción sin revisarla.

In [24]:
best_t2 = "mismo beneficiario → orden del documento"
base = best_t2.split("→ ")[-1]
cache = {"minilm": dict(zip(SV.statements, lp.encode([lp.statement_text(s) for s in SV.statements.values()])))}
for m, v in structure.items():
    cache[f"estructura: {m}"] = v
predicted, rows = {}, []
for cond, consequence, correct in t2_q:
    cands = [d for d in lp.duties(SV) if d != consequence]
    trigger = " ".join(c["trigger"] for c in lp.gating_conditions(SV).get(consequence, []))
    s = lp.t2_scores(SV, consequence, cands, trigger, base, cache)
    if best_t2.startswith("mismo beneficiario"):
        s = s + 1e6 * lp.same_beneficiary(SV, consequence, cands)
    top = cands[int(np.argmax(s))]
    predicted[consequence] = {"duties": [top], "source": "predicho"}
    debtor_ok = lp.direction(SV, top)[0] == lp.direction(SV, sorted(correct)[0])[0]
    clause_ok = SV.statements[top].get("clauseId") in {SV.statements[x].get("clauseId") for x in correct}
    rows.append({"condición": cond, "consecuencia": consequence, "predicho": top, "correcto": top in correct, "mismo deudor": debtor_ok, "misma cláusula": clause_ok})
display(Markdown(f"Método: **{best_t2}**"))
lp.table(rows)

Método: **mismo beneficiario → orden del documento**

| condición | consecuencia | predicho | correcto | mismo deudor | misma cláusula |
|---|---|---|---|---|---|
| C02 | obligation-7 | obligation-1 | False | True | True |
| C05 | obligation-16 | obligation-10 | True | True | True |
| C06 | obligation-14 | obligation-10 | True | True | True |
| C10 | right-10 | obligation-37 | True | True | True |
| C13 | right-12 | obligation-38 | True | True | True |
| C16 | obligation-46 | prohibition-17 | False | True | False |
| C16 | obligation-47 | prohibition-18 | False | True | False |
| C19 | obligation-39 | obligation-38 | True | True | True |
| C20 | obligation-13 | obligation-10 | True | True | True |

«Mismo deudor» sale siempre bien por construcción: con dos partes, exigir que el deber proteja
a la misma parte que la consecuencia fija también quién lo debía. Lo que falla es **qué** deber,
y con él la cláusula a la que apunta el camino.

## 8. El fichero aparte

Las predicciones de T2 para cada consecuencia con condición de SteelVault —los diez mejores
candidatos, con su puntuación— van a `infra/json/link_prediction/<contrato>.json`. Diez, porque
es donde la regla recomendada encuentra siempre el deber correcto. Cada una lleva la clase que le
da la anotación propuesta, para poder quitar las condiciones que no son un incumplimiento.

Usan el tipo `depends_on`, que ya existe, pero con un enunciado como destino, que el esquema no
admite: por eso se quedan fuera del grafo, marcadas como predichas, junto a su versión a nivel de
cláusula, que sí cabría.

In [25]:
annotated = {one[cons]: kind for _, cons, kind, _, _ in PROPOSAL if cons in one}
out = []
gates = lp.gating_conditions(SV)
for consequence in sorted(gates, key=lambda s: SV.position.get(s, (0, 0))):
    if consequence not in SV.statements:
        continue
    cands = [d for d in lp.duties(SV) if d != consequence]
    trigger = " ".join(c["trigger"] for c in gates[consequence])
    raw = lp.t2_scores(SV, consequence, cands, trigger, base, cache)
    s = raw + 1e6 * lp.same_beneficiary(SV, consequence, cands) if best_t2.startswith("mismo beneficiario") else raw
    for rank, i in enumerate(np.argsort(-s, kind="stable")[:10], start=1):
        out.append({
            "source": consequence, "type": "depends_on", "target": cands[i],
            "targetClause": SV.statements[cands[i]].get("clauseId"),
            "rank": rank, "score": round(float(raw[i]), 4), "method": best_t2,
            "conditions": [c["id"] for c in gates[consequence]], "status": "predicha, sin revisar",
            "annotatedClass": annotated.get(consequence, "sin anotar"),
        })
path = lp.RESULTS_DIR / f"{SV.stem}.json"
path.write_text(json.dumps({
    "documentId": SV.stem, "kgFingerprint": kg_prints["SteelVault"],
    "note": "Aristas predichas, fuera del grafo. depends_on de una consecuencia al deber cuyo incumplimiento la dispara; el esquema solo admite el destino a nivel de cláusula (targetClause).",
    "edges": out,
}, ensure_ascii=False, indent=1))
print(len(out), "aristas predichas →", path.name)

280 aristas predichas → root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement.json


## 9. Recomendación

**No adoptar embeddings de grafo para decidir a quién favorece una cláusula.** Dentro de un
contrato no tienen de dónde aprender quién carga con qué:

- **T1.** En los tres contratos, ningún modelo de grafo supera con claridad a la mayoría de la
  cláusula, y la regla del sujeto los supera a todos. TransH y TransR no arreglan nada, porque el
  muchos-a-uno no es el problema. La extracción ya acierta el 100% en SteelVault: no queda hueco
  que llenar, ni como predicción ni como auditoría (15% de falsas alarmas). Los siete
  enunciados sin parte de SteelVault no deben recibir una: seis no son de ninguna parte del
  contrato.
- **T3.** Como método único, la regla del número citado (MRR 0,74) supera a todos los modelos.
  Como complemento, NBFNet detrás de la regla sube a 0,82 en Ediets y 0,80 en Bellicum, sobre
  todo en `modifies`. Es la única tarea en la que un modelo de grafo aporta algo medible, y lo
  aporta sobre etiquetas sin revisar.
- **T2.** No hay ninguna arista de la que aprender. La cercanía estructural, con el filtro de
  beneficiario, encuentra bien los deberes de la misma cláusula y falla cuando el deber está en
  otra. Cifras provisionales hasta revisar la anotación.

**Adoptar, sin modelos entrenados:**

1. **A quién favorece.** El veredicto sin pesos de la sección 7 —unánime, dominante o depende de
   los pesos— en lugar del porcentaje. Cada veredicto lleva sus enunciados y sus aristas de
   parte. En SteelVault, 13 de las 15 cláusulas con enunciados unilaterales tienen un veredicto
   que ningún peso positivo cambia; las otras 2 son justo las que la barra decide por los pesos.
2. **Qué exposición genera.** Los caminos de exposición —deber de A → disparador → consecuencia a
   favor de B → cifras— y los de recorte, un derecho que `modifies` una cláusula donde su
   titular debe algo. En SteelVault, 9 caminos exponen al Affiliate y 1 a Equidata, y la
   exclusión de §11 recorta la única garantía de Equidata (§8.4).
3. **El vínculo consecuencia → deber.** Anotado en SteelVault (la propuesta de la sección 6, una
   vez revisada). En un contrato nuevo, la regla *mismo beneficiario → orden del documento*
   propone candidatos para revisar: el correcto está entre los tres primeros en 7 de 9 consultas
   y entre los diez primeros en todas. Se guarda fuera del grafo; su versión a nivel de
   cláusula cabría en el esquema actual como `depends_on` de enunciado a cláusula. La línea base
   que falta es un LLM: unos 0,04 dólares por las nueve consultas medidas, o 0,10 por las 28
   consecuencias de SteelVault. Está pendiente de aprobación.

**Dónde sí caben los embeddings de grafo:** entre contratos, sobre entidades que se repiten, como
el rol (comprador, proveedor) y el tipo de acción, no «party-2». Es el trabajo futuro del
encargo, y lo medido aquí lo apoya: lo que les falta a estos modelos dentro de un contrato es
justo lo que allí tendrían, muchos ejemplos de la misma entidad.